In [1]:
!pip install -q pandas==2.2.2 numpy==2.0.2
!pip install -q sentence-transformers==4.1.0 transformers==4.51.3 datasets accelerate faiss-cpu rank-bm25 scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 112.8 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import torch
import faiss
import random
import numpy as np
import pandas as pd

from sentence_transformers import CrossEncoder, SentenceTransformer, InputExample
from torch.utils.data import DataLoader
from rank_bm25 import BM25Okapi
from sklearn.model_selection import train_test_split

In [3]:
reranker_path = "/content/reranker.jsonl"

rows = []

with open(reranker_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

reranker_df = pd.DataFrame(rows)

print("Reranker df:", reranker_df.shape)
print(reranker_df.columns)
reranker_df.head()

Reranker df: (6752, 10)
Index(['id', 'query_id', 'query', 'candidate_passage', 'label', 'candidate_id',
       'citation_label', 'source', 'negative_type', 'audit_status'],
      dtype='object')


,id,query_id,query,candidate_passage,label,candidate_id,citation_label,source,negative_type,audit_status
0,rerank_noleak_balanced_000001,rerank_query_final_000001,18 yaşından küçük çocuk hakkında ORICON kaynağ...,18 yaşından küçük bir çocuk için DNA testi yap...,1,oricon_genel_001370,ORICON - Genel hukuk / sınıflandırma bekliyor ...,ORICON,None,no_eval_gold_leak_balanced_v3
1,rerank_noleak_balanced_000002,rerank_query_final_000001,18 yaşından küçük çocuk hakkında ORICON kaynağ...,İdarenin takdir yetkisi kamu yararı ve hizmet ...,0,oricon_genel_000975,ORICON - Genel hukuk / sınıflandırma bekliyor ...,ORICON,hard_negative_same_source_or_category,no_eval_gold_leak_balanced_v3
2,rerank_noleak_balanced_000003,rerank_query_final_000001,18 yaşından küçük çocuk hakkında ORICON kaynağ...,Satış bedelinin tamamının peşin ödenmesi hâlin...,0,oricon_genel_001651,ORICON - Genel hukuk / sınıflandırma bekliyor ...,ORICON,hard_negative_same_source_or_category,no_eval_gold_leak_balanced_v3
3,rerank_noleak_balanced_000004,rerank_query_final_000002,18 yaşını dolduran çocuk eğitimine hakkında OR...,18 yaşını dolduran çocuk eğitimine devam ediyo...,1,oricon_genel_001778,ORICON - Genel hukuk / sınıflandırma bekliyor ...,ORICON,None,no_eval_gold_leak_balanced_v3
4,rerank_noleak_balanced_000005,rerank_query_final_000002,18 yaşını dolduran çocuk eğitimine hakkında OR...,"İtiraz, Sulh Ceza Hakimliğine yapılmalıdır ve ...",0,oricon_genel_000063,ORICON - Genel hukuk / sınıflandırma bekliyor ...,ORICON,hard_negative_same_source_or_category,no_eval_gold_leak_balanced_v3


In [4]:
print("Label distribution:")
print(reranker_df["label"].value_counts(dropna=False))

if "negative_type" in reranker_df.columns:
    print("\nNegative type:")
    print(reranker_df["negative_type"].value_counts(dropna=False).head(20))

if "audit_status" in reranker_df.columns:
    print("\nAudit status:")
    print(reranker_df["audit_status"].value_counts(dropna=False))

Label distribution:
label
0    4299
1    2453
Name: count, dtype: int64

Negative type:
negative_type
hard_negative_same_source_or_category    4299
None                                     2453
Name: count, dtype: int64

Audit status:
audit_status
no_eval_gold_leak_balanced_v3    6752
Name: count, dtype: int64


In [5]:
def clean_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [6]:
train_examples = []

for _, row in reranker_df.iterrows():
    query = clean_text(row.get("query", ""))
    passage = clean_text(row.get("candidate_passage", ""))
    label = row.get("label", 0)

    try:
        label = float(label)
    except:
        label = 0.0

    if len(query) > 5 and len(passage) > 20:
        train_examples.append(
            InputExample(
                texts=[query, passage],
                label=label
            )
        )

print("Total reranker examples:", len(train_examples))
print("Sample texts:", train_examples[0].texts)
print("Sample label:", train_examples[0].label)

Total reranker examples: 6752
Sample texts: ['18 yaşından küçük çocuk hakkında ORICON kaynağına göre ne söylenebilir?', '18 yaşından küçük bir çocuk için DNA testi yaptırılabilmesi için anne ve babanın rızasına ihtiyaç vardır.']
Sample label: 1.0


In [7]:
train_data, val_data = train_test_split(
    train_examples,
    test_size=0.15,
    random_state=42
)

print("Train:", len(train_data))
print("Validation:", len(val_data))

Train: 5739
Validation: 1013


In [8]:
base_reranker_model = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

reranker_model = CrossEncoder(
    base_reranker_model,
    num_labels=1,
    max_length=512,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Loaded:", base_reranker_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loaded: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


In [9]:
train_dataloader = DataLoader(
    train_data,
    shuffle=True,
    batch_size=16
)

print("Train batches:", len(train_dataloader))

Train batches: 359


In [10]:
num_epochs = 1
warmup_steps = int(len(train_dataloader) * 0.1)

reranker_model.fit(
    train_dataloader=train_dataloader,
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path="./finetuned_legal_reranker",
    show_progress_bar=True
)

Token indices sequence length is longer than the specified maximum sequence length for this model (593 > 512). Running this sequence through the model will result in indexing errors
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss


In [11]:
test_pairs = [
    (
        "Kıdem tazminatı nedir?",
        "Kıdem tazminatı, işçinin her bir yıllık çalışması karşılığında 30 günlük brüt ücreti üzerinden hesaplanır ve iş akdinin sona ermesiyle ödenir."
    ),
    (
        "Kıdem tazminatı nedir?",
        "Aile konutu hakkında daha fazla bilgi edinmek isteyenler ilgili yazıyı inceleyebilirler."
    )
]

scores = reranker_model.predict(test_pairs)

for pair, score in zip(test_pairs, scores):
    print("\nQuery:", pair[0])
    print("Passage:", pair[1])
    print("Score:", score)


Query: Kıdem tazminatı nedir?
Passage: Kıdem tazminatı, işçinin her bir yıllık çalışması karşılığında 30 günlük brüt ücreti üzerinden hesaplanır ve iş akdinin sona ermesiyle ödenir.
Score: 6.428813

Query: Kıdem tazminatı nedir?
Passage: Aile konutu hakkında daha fazla bilgi edinmek isteyenler ilgili yazıyı inceleyebilirler.
Score: -6.6193337


In [12]:
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/CENG493_models/finetuned_legal_reranker"

os.makedirs("/content/drive/MyDrive/CENG493_models", exist_ok=True)

reranker_model.save(save_path)

print("Saved to:", save_path)

!ls -lh "/content/drive/MyDrive/CENG493_models/finetuned_legal_reranker"

Mounted at /content/drive
Saved to: /content/drive/MyDrive/CENG493_models/finetuned_legal_reranker
total 470M
-rw------- 1 root root  851 May 30 14:53 config.json
-rw------- 1 root root 449M May 30 14:53 model.safetensors
-rw------- 1 root root  15K May 30 14:53 README.md
-rw------- 1 root root 4.9M May 30 14:53 sentencepiece.bpe.model
-rw------- 1 root root  964 May 30 14:53 special_tokens_map.json
-rw------- 1 root root 1.2K May 30 14:53 tokenizer_config.json
-rw------- 1 root root  17M May 30 14:53 tokenizer.json


In [13]:
path = "/content/drive/MyDrive/CENG493_models/finetuned_legal_reranker"

print("exists:", os.path.exists(path))
print("config:", os.path.exists(os.path.join(path, "config.json")))
print("files:", os.listdir(path))

exists: True
config: True
files: ['config.json', 'model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'sentencepiece.bpe.model', 'tokenizer.json', 'README.md']


In [14]:
test_reranker = CrossEncoder(
    "/content/drive/MyDrive/CENG493_models/finetuned_legal_reranker",
    device="cpu"
)

print("Reload OK")

Reload OK


In [15]:
corpus_path = "/content/corpus.jsonl"

corpus = []

with open(corpus_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            corpus.append(json.loads(line))

documents = []
metadatas = []

for item in corpus:
    text = clean_text(item.get("text", ""))

    if len(text) > 20:
        documents.append(text)
        metadatas.append({
            "doc_id": item.get("id"),
            "title": item.get("title", ""),
            "metadata": item.get("metadata", {})
        })

print("Corpus:", len(corpus))
print("Usable documents:", len(documents))

Corpus: 7579
Usable documents: 7579


In [16]:
base_embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(base_embedding_model_name)

doc_embeddings = embedding_model.encode(
    documents,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

doc_embeddings = np.asarray(doc_embeddings).astype("float32")

index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

print("Embedding shape:", doc_embeddings.shape)
print("FAISS index:", index.ntotal)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Embedding shape: (7579, 384)
FAISS index: 7579


In [17]:
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

print("BM25 corpus:", len(tokenized_docs))

BM25 corpus: 7579


In [18]:
def dense_retrieve_base(query, k=100):
    q_emb = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(q_emb, k)

    results = []

    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        results.append({
            "idx": int(idx),
            "rank": rank,
            "dense_score": float(score),
            "text": documents[idx],
            "meta": metadatas[idx],
            "method": "dense"
        })

    return results


def bm25_retrieve_base(query, k=100):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:k]

    results = []

    for rank, idx in enumerate(top_idx, start=1):
        results.append({
            "idx": int(idx),
            "rank": rank,
            "bm25_score": float(scores[idx]),
            "text": documents[idx],
            "meta": metadatas[idx],
            "method": "bm25"
        })

    return results


def hybrid_candidates_base(query, dense_k=100, bm25_k=100, dense_weight=0.5, bm25_weight=0.5):
    dense_docs = dense_retrieve_base(query, k=dense_k)
    bm25_docs = bm25_retrieve_base(query, k=bm25_k)

    candidates = {}

    for doc in dense_docs:
        idx = doc["idx"]
        candidates[idx] = doc.copy()
        candidates[idx]["fusion_score"] = candidates[idx].get("fusion_score", 0) + dense_weight * (1 / doc["rank"])

    for doc in bm25_docs:
        idx = doc["idx"]

        if idx not in candidates:
            candidates[idx] = doc.copy()
            candidates[idx]["fusion_score"] = 0

        candidates[idx]["fusion_score"] += bm25_weight * (1 / doc["rank"])

    ranked = sorted(
        candidates.values(),
        key=lambda x: x["fusion_score"],
        reverse=True
    )

    return ranked


def finetuned_rerank(query, candidates, top_k=10, rerank_k=80):
    candidates = candidates[:rerank_k]

    pairs = [
        (query, cand["text"])
        for cand in candidates
    ]

    scores = reranker_model.predict(
        pairs,
        batch_size=32,
        show_progress_bar=False
    )

    reranked = []

    for cand, score in zip(candidates, scores):
        new_cand = cand.copy()
        new_cand["ft_reranker_score"] = float(score)
        reranked.append(new_cand)

    reranked = sorted(
        reranked,
        key=lambda x: x["ft_reranker_score"],
        reverse=True
    )

    return reranked[:top_k]


def retrieve_with_ft_reranker(query, top_k=10):
    candidates = hybrid_candidates_base(
        query,
        dense_k=100,
        bm25_k=100,
        dense_weight=0.5,
        bm25_weight=0.5
    )

    retrieved = finetuned_rerank(
        query,
        candidates,
        top_k=top_k,
        rerank_k=80
    )

    return retrieved

In [19]:
gold_path = "/content/gold_benchmark.json"

gold_df = pd.read_json(gold_path)

eval_df = gold_df.sample(
    n=min(100, len(gold_df)),
    random_state=42
).reset_index(drop=True)

print("Gold size:", len(gold_df))
print("Eval size:", len(eval_df))

Gold size: 240
Eval size: 100


In [20]:
def extract_gold_ids(gold_sources):
    ids = set()

    if isinstance(gold_sources, list):
        for item in gold_sources:
            if isinstance(item, dict):
                for key in ["doc_id", "id", "chunk_id", "source_id"]:
                    if key in item:
                        ids.add(str(item[key]))
            else:
                ids.add(str(item))

    elif isinstance(gold_sources, dict):
        for key in ["doc_id", "id", "chunk_id", "source_id"]:
            if key in gold_sources:
                ids.add(str(gold_sources[key]))

    elif isinstance(gold_sources, str):
        ids.add(str(gold_sources))

    return ids


def source_match(retrieved_id, gold_ids):
    retrieved_id = str(retrieved_id)

    for gid in gold_ids:
        gid = str(gid)

        if retrieved_id == gid:
            return True

        if retrieved_id in gid or gid in retrieved_id:
            return True

    return False


def hit_at_k(retrieved, gold_ids, k):
    return int(any(source_match(d["meta"]["doc_id"], gold_ids) for d in retrieved[:k]))


def reciprocal_rank(retrieved, gold_ids):
    for rank, d in enumerate(retrieved, start=1):
        if source_match(d["meta"]["doc_id"], gold_ids):
            return 1 / rank
    return 0


def ndcg_at_k(retrieved, gold_ids, k):
    dcg = 0.0

    for i, d in enumerate(retrieved[:k], start=1):
        rel = 1 if source_match(d["meta"]["doc_id"], gold_ids) else 0

        if rel:
            dcg += 1 / np.log2(i + 1)

    ideal_hits = min(len(gold_ids), k)

    if ideal_hits == 0:
        return 0.0

    idcg = sum(1 / np.log2(i + 1) for i in range(1, ideal_hits + 1))

    return dcg / idcg if idcg > 0 else 0.0

In [21]:
ft_reranker_rows = []

for i, row in eval_df.iterrows():
    question = row["question"]
    gold_ids = extract_gold_ids(row["gold_sources"])

    retrieved = retrieve_with_ft_reranker(
        question,
        top_k=10
    )

    ft_reranker_rows.append({
        "question": question,
        "Recall@5": hit_at_k(retrieved, gold_ids, 5),
        "Recall@10": hit_at_k(retrieved, gold_ids, 10),
        "MRR": reciprocal_rank(retrieved, gold_ids),
        "nDCG@10": ndcg_at_k(retrieved, gold_ids, 10)
    })

    if (i + 1) % 10 == 0:
        print(f"Processed {i+1}/{len(eval_df)}")

ft_reranker_results_df = pd.DataFrame(ft_reranker_rows)

ft_reranker_metrics = {
    "Recall@5": ft_reranker_results_df["Recall@5"].mean(),
    "Recall@10": ft_reranker_results_df["Recall@10"].mean(),
    "MRR": ft_reranker_results_df["MRR"].mean(),
    "nDCG@10": ft_reranker_results_df["nDCG@10"].mean()
}

ft_reranker_metrics

Processed 10/100
Processed 20/100
Processed 30/100
Processed 40/100
Processed 50/100
Processed 60/100
Processed 70/100
Processed 80/100
Processed 90/100
Processed 100/100


{'Recall@5': np.float64(0.76),
 'Recall@10': np.float64(0.79),
 'MRR': np.float64(0.7242499999999998),
 'nDCG@10': np.float64(0.7525309666675049)}

In [22]:
reranker_comparison = pd.DataFrame([
    {
        "System": "Base Hybrid Retrieval",
        "Recall@5": 0.735,
        "Recall@10": 0.765,
        "MRR": 0.658248,
        "nDCG@10": 0.670770
    },
    {
        "System": "Hybrid + Base Cross-Encoder Reranker",
        "Recall@5": 0.760,
        "Recall@10": 0.770,
        "MRR": 0.682589,
        "nDCG@10": 0.689678
    },
    {
        "System": "Hybrid + Fine-tuned Legal Reranker",
        **ft_reranker_metrics
    }
])

reranker_comparison

,System,Recall@5,Recall@10,MRR,nDCG@10
0,Base Hybrid Retrieval,0.735,0.765,0.658248,0.670770
1,Hybrid + Base Cross-Encoder Reranker,0.760,0.770,0.682589,0.689678
2,Hybrid + Fine-tuned Legal Reranker,0.760,0.790,0.724250,0.752531
